# 04 — Avaliação

**Tech Challenge — Fase 2 | POSTECH Data Analytics | Grupo 08**

Escopo: métricas, ajuste do limiar, importância das variáveis, modelo final e implicações de negócio.

**Entradas:** artefatos de `02_preprocessamento.ipynb` e `03_modelagem.ipynb`.
**Saída:** `data/processed/modelo_hgb_final_12_meses.joblib` (pipeline + metadados + função de classificação).

> As saídas foram geradas no Colab original. As células "carregar" e "resumo" são novas e ficam sem saída até a execução.

In [ ]:
# Configuração comum a todos os notebooks do projeto
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEMENTE = 42  # semente fixa usada em todo o projeto

# Funciona rodando a partir de notebooks/ ou da raiz do repositório.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"              # arquivos brutos (não versionados)
PROCESSED = RAIZ / "data" / "processed"  # artefatos intermediários (não versionados)
PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
import joblib

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix,
)

# Artefatos dos notebooks 02 e 03
base_total_trat = joblib.load(PROCESSED / "base_total_trat.joblib")
CREDIT_FILTRADO = joblib.load(PROCESSED / "credit_filtrado.joblib")
BASE_MODELO = joblib.load(PROCESSED / "base_modelo.joblib")

_val = joblib.load(PROCESSED / "validacao_12_meses.joblib")
X_limiar = _val["X_limiar"]
y_limiar = _val["y_limiar"]
grupos_limiar = _val["grupos_limiar"]
ids_limiar = _val["ids_limiar"]
plano_folds = _val["plano_folds"]
diagnosticos_internos = _val["diagnosticos_internos"]
modelo_base = _val["modelo_base"]

_res = joblib.load(PROCESSED / "resultados_modelagem.joblib")
COMPARACAO_PROTOCOLOS = _res["COMPARACAO_PROTOCOLOS"]
MEDIAS_CV = _res["MEDIAS_CV"]
DESVIOS_CV = _res["DESVIOS_CV"]
MEDIAS_JANELAS = _res["MEDIAS_JANELAS"]
MEDIAS_BUSCA_HGB = _res["MEDIAS_BUSCA_HGB"]
RESULTADOS_BUSCA_HGB_FOLDS = _res["RESULTADOS_BUSCA_HGB_FOLDS"]

# Configuração da validação de 12 meses
FOLDS_EXTERNOS = 5
FOLDS_INTERNOS = 3
LIMIAR_PADRAO = 0.50

print(f"Clientes (12 meses): {len(y_limiar):,} | Classe 1: {int(y_limiar.sum()):,} ({100 * y_limiar.mean():.2f}%)")

## 1. Resumo das métricas de validação

Recupera os resultados do notebook 03. Métricas da classe 1 (atraso 60+) em folds separados por perfil; a acurácia é pouco informativa porque o `Dummy` já acerta ~98%.

In [ ]:
# Resumo dos resultados de validação gerados no notebook 03
print("Médias dos 5 folds por perfil (base de 29.029 clientes):")
display(MEDIAS_CV.round(4))

print("\nDesvios padrão dos 5 folds:")
display(DESVIOS_CV.round(4))

print("\nHist Gradient Boosting por cenário de janela:")
display(MEDIAS_JANELAS.round(4))

print("\nModelo atual x busca de hiperparâmetros (12 meses):")
display(MEDIAS_BUSCA_HGB.round(4))

## 2. Limiar de decisão

O limiar padrão de 0,50 é comparado a limiares escolhidos em validação interna por F1, F2 e F3. O teste externo nunca participa da escolha. Mais recall significa analisar mais clientes: é uma decisão de negócio (custo de análise x custo do atraso).

In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix
)

# Funções auxiliares da validação (mesmas do notebook 03)
# ============================================================
# 2. Funções auxiliares
# ============================================================

def validar_divisao(y, grupos, treino, teste, descricao):
    """Valida classes e ausência de perfis compartilhados."""

    compartilhados = (
        set(grupos.iloc[treino])
        & set(grupos.iloc[teste])
    )

    if compartilhados:
        raise ValueError(
            f"{descricao}: existem perfis compartilhados."
        )

    if (
        y.iloc[treino].nunique() != 2
        or y.iloc[teste].nunique() != 2
    ):
        raise ValueError(
            f"{descricao}: treino e validação/teste "
            "precisam conter as duas classes."
        )


def probabilidades_classe_1(modelo, X):
    indice = list(modelo.classes_).index(1)
    return modelo.predict_proba(X)[:, indice]


def escolher_limiar_f1(y, probabilidades):
    """
    Escolhe o limiar com maior F1 nas previsões internas
    fora da amostra. Em empate, escolhe o maior limiar.
    """

    precisao, recall, limiares = precision_recall_curve(
        y, probabilidades
    )

    # O último ponto da curva não possui limiar correspondente.
    precisao = precisao[:-1]
    recall = recall[:-1]

    f1 = np.divide(
        2 * precisao * recall,
        precisao + recall,
        out=np.zeros_like(precisao),
        where=(precisao + recall) > 0
    )

    melhor_f1 = f1.max()

    empatados = np.flatnonzero(
        np.isclose(f1, melhor_f1, rtol=0, atol=1e-12)
    )

    indice = empatados[-1]

    return {
        "LIMIAR_AJUSTADO": float(limiares[indice]),
        "F1_INTERNO": float(f1[indice]),
        "PRECISAO_INTERNA": float(precisao[indice]),
        "RECALL_INTERNO": float(recall[indice])
    }


def avaliar_limiar(y, probabilidades, limiar):
    previsoes = (probabilidades >= limiar).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, previsoes, labels=[0, 1]
    ).ravel()

    return {
        "ACURACIA": accuracy_score(y, previsoes),
        "ACURACIA_BALANCEADA": balanced_accuracy_score(
            y, previsoes
        ),
        "PRECISAO_CLASSE_1": precision_score(
            y, previsoes, zero_division=0
        ),
        "RECALL_CLASSE_1": recall_score(
            y, previsoes, zero_division=0
        ),
        "F1_CLASSE_1": f1_score(
            y, previsoes, zero_division=0
        ),
        "AVERAGE_PRECISION": average_precision_score(
            y, probabilidades
        ),
        "ROC_AUC": roc_auc_score(y, probabilidades),
        "VERDADEIROS_NEGATIVOS": int(tn),
        "FALSOS_POSITIVOS": int(fp),
        "FALSOS_NEGATIVOS": int(fn),
        "VERDADEIROS_POSITIVOS": int(tp),
        "CLIENTES_SINALIZADOS": int(fp + tp)
    }

In [ ]:
# AJUSTE DO LIMIAR DE CLASSIFICAÇÃO DO HIST GRADIENT BOOSTING

# ============================================================
# 4. Ajustar limiar internamente e avaliar externamente
# ============================================================

resultados = []
limiares_escolhidos = []
previsoes_externas = []

for fold, treino, teste, divisoes_internas in plano_folds:
    print(f"\nFold externo {fold}/{FOLDS_EXTERNOS}")

    X_treino = X_limiar.iloc[treino].reset_index(drop=True)
    y_treino = y_limiar.iloc[treino].reset_index(drop=True)

    X_teste = X_limiar.iloc[teste]
    y_teste = y_limiar.iloc[teste]

    # Cada cliente do treino externo recebe uma previsão
    # de um modelo que não treinou em seu perfil.
    probabilidades_internas = np.full(
        len(y_treino), np.nan
    )

    for fold_interno, (tr_int, val_int) in enumerate(
        divisoes_internas, start=1
    ):
        print(
            f"  Treinando fold interno "
            f"{fold_interno}/{FOLDS_INTERNOS}"
        )

        modelo_interno = clone(modelo_base)

        modelo_interno.fit(
            X_treino.iloc[tr_int],
            y_treino.iloc[tr_int]
        )

        probabilidades_internas[val_int] = (
            probabilidades_classe_1(
                modelo_interno,
                X_treino.iloc[val_int]
            )
        )

    if not np.isfinite(probabilidades_internas).all():
        raise ValueError(
            "Existem previsões internas ausentes ou inválidas."
        )

    escolha = escolher_limiar_f1(
        y_treino, probabilidades_internas
    )

    limiar_ajustado = escolha["LIMIAR_AJUSTADO"]

    # Agora treina com todo o treino externo.
    # O teste externo continua fora de todos os ajustes.
    modelo_externo = clone(modelo_base)
    modelo_externo.fit(X_treino, y_treino)

    probabilidades_teste = probabilidades_classe_1(
        modelo_externo, X_teste
    )

    for estrategia, limiar in [
        ("Limiar padrão", LIMIAR_PADRAO),
        ("Limiar ajustado", limiar_ajustado)
    ]:
        resultados.append({
            "FOLD": fold,
            "ESTRATEGIA": estrategia,
            "LIMIAR": limiar,
            **avaliar_limiar(
                y_teste, probabilidades_teste, limiar
            )
        })

    limiares_escolhidos.append({
        "FOLD": fold,
        "CLIENTES_TREINO": len(treino),
        "CLIENTES_TESTE": len(teste),
        "CLASSE_1_TREINO": int(y_treino.sum()),
        "CLASSE_1_TESTE": int(y_teste.sum()),
        "PERFIS_COMPARTILHADOS": 0,
        **escolha
    })

    previsoes_externas.append(pd.DataFrame({
        "ID": ids_limiar.iloc[teste].to_numpy(),
        "GRUPO_PERFIL": grupos_limiar.iloc[teste].to_numpy(),
        "FOLD": fold,
        "ALVO": y_teste.to_numpy(),
        "PROBABILIDADE_CLASSE_1": probabilidades_teste,
        "LIMIAR_AJUSTADO": limiar_ajustado,
        "PREVISAO_PADRAO": (
            probabilidades_teste >= LIMIAR_PADRAO
        ).astype(int),
        "PREVISAO_AJUSTADA": (
            probabilidades_teste >= limiar_ajustado
        ).astype(int)
    }))

    print(f"  Limiar escolhido: {limiar_ajustado:.4f}")


# ============================================================
# 5. Resultados no próprio ambiente Python
# ============================================================

RESULTADOS_LIMIAR_FOLDS = pd.DataFrame(resultados)

LIMIARES_POR_FOLD = pd.DataFrame(limiares_escolhidos)

DIAGNOSTICO_FOLDS_INTERNOS = pd.DataFrame(
    diagnosticos_internos
)

PREVISOES_EXTERNAS_LIMIAR = pd.concat(
    previsoes_externas, ignore_index=True
)

metricas = [
    "ACURACIA",
    "ACURACIA_BALANCEADA",
    "PRECISAO_CLASSE_1",
    "RECALL_CLASSE_1",
    "F1_CLASSE_1",
    "AVERAGE_PRECISION",
    "ROC_AUC"
]

MEDIAS_LIMIAR = (
    RESULTADOS_LIMIAR_FOLDS
    .groupby("ESTRATEGIA", sort=False)[metricas]
    .mean()
)

DESVIOS_LIMIAR = (
    RESULTADOS_LIMIAR_FOLDS
    .groupby("ESTRATEGIA", sort=False)[metricas]
    .std()
)

RESUMO_LIMIARES = (
    LIMIARES_POR_FOLD["LIMIAR_AJUSTADO"]
    .agg(["mean", "std", "min", "max"])
    .rename(index={
        "mean": "MEDIA",
        "std": "DESVIO_PADRAO",
        "min": "MINIMO",
        "max": "MAXIMO"
    })
    .to_frame("LIMIAR_AJUSTADO")
)

# Diferenças pareadas: ajustado menos padrão no mesmo fold.
padrao = (
    RESULTADOS_LIMIAR_FOLDS
    .query("ESTRATEGIA == 'Limiar padrão'")
    .set_index("FOLD")[metricas]
)

ajustado = (
    RESULTADOS_LIMIAR_FOLDS
    .query("ESTRATEGIA == 'Limiar ajustado'")
    .set_index("FOLD")[metricas]
)

DIFERENCAS_LIMIAR_FOLDS = ajustado - padrao

RESUMO_DIFERENCAS_LIMIAR = (
    DIFERENCAS_LIMIAR_FOLDS
    .agg(["mean", "std", "min", "max"])
    .T
    .rename(columns={
        "mean": "DIFERENCA_MEDIA",
        "std": "DESVIO_PADRAO",
        "min": "MINIMO",
        "max": "MAXIMO"
    })
)

colunas_contagem = [
    "VERDADEIROS_NEGATIVOS",
    "FALSOS_POSITIVOS",
    "FALSOS_NEGATIVOS",
    "VERDADEIROS_POSITIVOS",
    "CLIENTES_SINALIZADOS"
]

# Cada cliente aparece uma única vez como teste externo.
CONTAGENS_EXTERNAS_LIMIAR = (
    RESULTADOS_LIMIAR_FOLDS
    .groupby("ESTRATEGIA", sort=False)[colunas_contagem]
    .sum()
)

print("\nLIMIARES ESCOLHIDOS E DIAGNÓSTICO EXTERNO:")
display(LIMIARES_POR_FOLD.round(4))

print("\nMÉDIAS DOS CINCO FOLDS:")
display(MEDIAS_LIMIAR.round(4))

print("\nDESVIOS PADRÃO DOS CINCO FOLDS:")
display(DESVIOS_LIMIAR.round(4))

print("\nVARIAÇÃO DOS LIMIARES:")
display(RESUMO_LIMIARES.round(4))

print("\nDIFERENÇAS: AJUSTADO MENOS PADRÃO:")
display(RESUMO_DIFERENCAS_LIMIAR.round(4))

print("\nCONTAGENS SOMADAS DOS TESTES EXTERNOS:")
display(CONTAGENS_EXTERNAS_LIMIAR)

for estrategia, linha in CONTAGENS_EXTERNAS_LIMIAR.iterrows():
    matriz = pd.DataFrame(
        [
            [
                linha["VERDADEIROS_NEGATIVOS"],
                linha["FALSOS_POSITIVOS"]
            ],
            [
                linha["FALSOS_NEGATIVOS"],
                linha["VERDADEIROS_POSITIVOS"]
            ]
        ],
        index=["Real: classe 0", "Real: classe 1"],
        columns=["Previsto: classe 0", "Previsto: classe 1"]
    )

    print(f"\nMATRIZ DE CONFUSÃO — {estrategia}:")
    display(matriz)


Fold externo 1/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar escolhido: 0.7365

Fold externo 2/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar escolhido: 0.7502

Fold externo 3/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar escolhido: 0.6170

Fold externo 4/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar escolhido: 0.6908

Fold externo 5/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar escolhido: 0.7283

LIMIARES ESCOLHIDOS E DIAGNÓSTICO EXTERNO:


   FOLD  CLIENTES_TREINO  CLIENTES_TESTE  CLASSE_1_TREINO  CLASSE_1_TESTE  \
0     1            17455            4276              300              70   
1     2            17483            4248              276              94   
2     3            17222            4509              292              78   
3     4            17495            4236              304              66   
4     5            17269            4462              308              62   

   PERFIS_COMPARTILHADOS  LIMIAR_AJUSTADO  F1_INTERNO  PRECISAO_INTERNA  \
0                      0           0.7365      0.1399            0.2326   
1                      0           0.7502      0.1504            0.3253   
2                      0           0.6170      0.1721            0.1948   
3                      0           0.6908      0.1316            0.2857   
4                      0           0.7283      0.1867            0.2958   

   RECALL_INTERNO  
0          0.1000  
1          0.0978  
2          0.1541  
3     


MÉDIAS DOS CINCO FOLDS:


                 ACURACIA  ACURACIA_BALANCEADA  PRECISAO_CLASSE_1  \
ESTRATEGIA                                                          
Limiar padrão      0.9613               0.5856             0.1185   
Limiar ajustado    0.9774               0.5642             0.2273   

                 RECALL_CLASSE_1  F1_CLASSE_1  AVERAGE_PRECISION  ROC_AUC  
ESTRATEGIA                                                                 
Limiar padrão             0.1967       0.1455             0.1223   0.6212  
Limiar ajustado           0.1364       0.1661             0.1223   0.6212  


DESVIOS PADRÃO DOS CINCO FOLDS:


...

                 VERDADEIROS_NEGATIVOS  FALSOS_POSITIVOS  FALSOS_NEGATIVOS  \
ESTRATEGIA                                                                   
Limiar padrão                    20821               540               298   
Limiar ajustado                  21190               171               320   

                 VERDADEIROS_POSITIVOS  CLIENTES_SINALIZADOS  
ESTRATEGIA                                                    
Limiar padrão                       72                   612  
Limiar ajustado                     50                   221  


MATRIZ DE CONFUSÃO — Limiar padrão:


                Previsto: classe 0  Previsto: classe 1
Real: classe 0               20821                 540
Real: classe 1                 298                  72


MATRIZ DE CONFUSÃO — Limiar ajustado:


                Previsto: classe 0  Previsto: classe 1
Real: classe 0               21190                 171
Real: classe 1                 320                  50

In [ ]:
#COMPARA LIMIARES

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve, fbeta_score

# ============================================================
# 1. Verificar objetos da célula anterior
# ============================================================

necessarios = [
    "X_limiar",
    "y_limiar",
    "grupos_limiar",
    "ids_limiar",
    "modelo_base",
    "plano_folds",
    "validar_divisao",
    "probabilidades_classe_1",
    "avaliar_limiar"
]

faltantes = [
    nome for nome in necessarios
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        "Execute primeiro a célula anterior de ajuste do limiar. "
        f"Objetos ausentes: {faltantes}"
    )

BETAS = [1, 2, 3]
LIMIAR_PADRAO = 0.50

print(f"Clientes: {len(y_limiar):,}")
print(f"Classe 1: {int(y_limiar.sum()):,}")
print(f"Folds externos: {len(plano_folds)}")


# ============================================================
# 2. Escolher o limiar para cada F-beta
# ============================================================

def selecionar_limiares_fbeta(y, probabilidades):
    """
    Usa somente as previsões internas fora da amostra.
    Em empate no F-beta, escolhe o maior limiar.
    """

    precisao, recall, limiares = precision_recall_curve(
        y, probabilidades
    )

    # O último ponto não possui limiar correspondente.
    precisao = precisao[:-1]
    recall = recall[:-1]

    escolhas = []

    for beta in BETAS:
        beta_quadrado = beta ** 2

        numerador = (
            (1 + beta_quadrado) * precisao * recall
        )
        denominador = beta_quadrado * precisao + recall

        valores_fbeta = np.divide(
            numerador,
            denominador,
            out=np.zeros_like(precisao),
            where=denominador > 0
        )

        melhor_valor = valores_fbeta.max()

        empatados = np.flatnonzero(
            np.isclose(
                valores_fbeta,
                melhor_valor,
                rtol=0,
                atol=1e-12
            )
        )

        indice = empatados[-1]

        escolhas.append({
            "ESTRATEGIA": f"Limiar por F{beta}",
            "BETA": beta,
            "LIMIAR": float(limiares[indice]),
            "FBETA_INTERNO": float(valores_fbeta[indice]),
            "PRECISAO_INTERNA": float(precisao[indice]),
            "RECALL_INTERNO": float(recall[indice])
        })

    return escolhas


# ============================================================
# 3. Treinamento interno e avaliação externa
# ============================================================

registros_resultados = []
registros_limiares = []
registros_previsoes = []

for fold, treino, teste, divisoes_internas in plano_folds:
    print(f"\nFold externo {fold}/{len(plano_folds)}")

    validar_divisao(
        y_limiar,
        grupos_limiar,
        treino,
        teste,
        f"Fold externo {fold}"
    )

    X_treino = X_limiar.iloc[treino].reset_index(drop=True)
    y_treino = y_limiar.iloc[treino].reset_index(drop=True)
    grupos_treino = (
        grupos_limiar.iloc[treino].reset_index(drop=True)
    )

    X_teste = X_limiar.iloc[teste]
    y_teste = y_limiar.iloc[teste]

    probabilidades_internas = np.full(
        len(y_treino), np.nan
    )
    vezes_validado = np.zeros(len(y_treino), dtype=int)

    # Mesmas divisões internas para F1, F2 e F3.
    for fold_interno, (tr_int, val_int) in enumerate(
        divisoes_internas, start=1
    ):
        validar_divisao(
            y_treino,
            grupos_treino,
            tr_int,
            val_int,
            f"Fold externo {fold}, interno {fold_interno}"
        )

        print(
            f"  Treinando fold interno "
            f"{fold_interno}/{len(divisoes_internas)}"
        )

        modelo_interno = clone(modelo_base)
        modelo_interno.set_params(
            modelo__early_stopping=False
        )

        modelo_interno.fit(
            X_treino.iloc[tr_int],
            y_treino.iloc[tr_int]
        )

        probabilidades_internas[val_int] = (
            probabilidades_classe_1(
                modelo_interno,
                X_treino.iloc[val_int]
            )
        )

        vezes_validado[val_int] += 1

    if (
        not np.isfinite(probabilidades_internas).all()
        or not np.all(vezes_validado == 1)
    ):
        raise ValueError(
            "Cada cliente do treino externo deve receber "
            "exatamente uma previsão interna fora da amostra."
        )

    escolhas = selecionar_limiares_fbeta(
        y_treino,
        probabilidades_internas
    )

    # Um único modelo externo para todos os critérios.
    modelo_externo = clone(modelo_base)
    modelo_externo.set_params(
        modelo__early_stopping=False
    )

    modelo_externo.fit(X_treino, y_treino)

    probabilidades_teste = probabilidades_classe_1(
        modelo_externo,
        X_teste
    )

    estrategias = [
        {
            "ESTRATEGIA": "Limiar padrão",
            "LIMIAR": LIMIAR_PADRAO
        },
        *escolhas
    ]

    for escolha in escolhas:
        registros_limiares.append({
            "FOLD": fold,
            "CLIENTES_TREINO": len(treino),
            "CLIENTES_TESTE": len(teste),
            "CLASSE_1_TREINO": int(y_treino.sum()),
            "CLASSE_1_TESTE": int(y_teste.sum()),
            "PERFIS_COMPARTILHADOS": 0,
            **escolha
        })

        print(
            f"  {escolha['ESTRATEGIA']}: "
            f"{escolha['LIMIAR']:.4f}"
        )

    for escolha in estrategias:
        estrategia = escolha["ESTRATEGIA"]
        limiar = escolha["LIMIAR"]

        previsoes = (
            probabilidades_teste >= limiar
        ).astype(int)

        avaliacao = avaliar_limiar(
            y_teste,
            probabilidades_teste,
            limiar
        )

        tp = avaliacao["VERDADEIROS_POSITIVOS"]
        sinalizados = avaliacao["CLIENTES_SINALIZADOS"]

        registros_resultados.append({
            "FOLD": fold,
            "ESTRATEGIA": estrategia,
            "LIMIAR": limiar,
            "CLIENTES_TESTE": len(teste),
            "CLASSE_1_TESTE": int(y_teste.sum()),
            **avaliacao,
            "F2_CLASSE_1": fbeta_score(
                y_teste,
                previsoes,
                beta=2,
                zero_division=0
            ),
            "F3_CLASSE_1": fbeta_score(
                y_teste,
                previsoes,
                beta=3,
                zero_division=0
            ),
            "PERCENTUAL_ENCAMINHADOS": (
                100 * sinalizados / len(teste)
            ),
            # Indefinido quando nenhum atraso é identificado.
            "ANALISES_POR_ATRASO_IDENTIFICADO": (
                sinalizados / tp if tp > 0 else np.nan
            )
        })

        registros_previsoes.append(pd.DataFrame({
            "ID": ids_limiar.iloc[teste].to_numpy(),
            "GRUPO_PERFIL": (
                grupos_limiar.iloc[teste].to_numpy()
            ),
            "FOLD": fold,
            "ESTRATEGIA": estrategia,
            "ALVO": y_teste.to_numpy(),
            "PROBABILIDADE_CLASSE_1": probabilidades_teste,
            "LIMIAR": limiar,
            "PREVISAO": previsoes
        }))


# ============================================================
# 4. Resumir métricas e limiares
# ============================================================

RESULTADOS_FBETAS_FOLDS = pd.DataFrame(
    registros_resultados
)

LIMIARES_FBETAS_FOLDS = pd.DataFrame(
    registros_limiares
)

PREVISOES_EXTERNAS_FBETAS = pd.concat(
    registros_previsoes,
    ignore_index=True
)

ordem = [
    "Limiar padrão",
    "Limiar por F1",
    "Limiar por F2",
    "Limiar por F3"
]

metricas = [
    "ACURACIA",
    "ACURACIA_BALANCEADA",
    "PRECISAO_CLASSE_1",
    "RECALL_CLASSE_1",
    "F1_CLASSE_1",
    "F2_CLASSE_1",
    "F3_CLASSE_1",
    "AVERAGE_PRECISION",
    "ROC_AUC",
    "PERCENTUAL_ENCAMINHADOS"
]

MEDIAS_FBETAS = (
    RESULTADOS_FBETAS_FOLDS
    .groupby("ESTRATEGIA")[metricas]
    .mean()
    .reindex(ordem)
)

DESVIOS_FBETAS = (
    RESULTADOS_FBETAS_FOLDS
    .groupby("ESTRATEGIA")[metricas]
    .std()
    .reindex(ordem)
)

RESUMO_LIMIARES_FBETAS = (
    LIMIARES_FBETAS_FOLDS
    .groupby("ESTRATEGIA")["LIMIAR"]
    .agg(["mean", "std", "min", "max"])
    .rename(columns={
        "mean": "MEDIA",
        "std": "DESVIO_PADRAO",
        "min": "MINIMO",
        "max": "MAXIMO"
    })
    .reindex(ordem[1:])
)

# Diferenças pareadas em relação ao padrão.
# Cada comparação utiliza exatamente o mesmo teste externo.
padrao = (
    RESULTADOS_FBETAS_FOLDS
    .loc[
        RESULTADOS_FBETAS_FOLDS["ESTRATEGIA"]
        .eq("Limiar padrão")
    ]
    .set_index("FOLD")[metricas]
)

diferencas = []

for estrategia in ordem[1:]:
    ajustado = (
        RESULTADOS_FBETAS_FOLDS
        .loc[
            RESULTADOS_FBETAS_FOLDS["ESTRATEGIA"]
            .eq(estrategia)
        ]
        .set_index("FOLD")[metricas]
    )

    diferenca = ajustado.subtract(padrao)
    diferenca["ESTRATEGIA"] = estrategia

    diferencas.append(diferenca.reset_index())

DIFERENCAS_FBETAS_FOLDS = pd.concat(
    diferencas, ignore_index=True
)

RESUMO_DIFERENCAS_FBETAS = (
    DIFERENCAS_FBETAS_FOLDS
    .melt(
        id_vars=["FOLD", "ESTRATEGIA"],
        value_vars=metricas,
        var_name="METRICA",
        value_name="DIFERENCA"
    )
    .groupby(["ESTRATEGIA", "METRICA"])["DIFERENCA"]
    .agg(["mean", "std", "min", "max"])
    .rename(columns={
        "mean": "DIFERENCA_MEDIA",
        "std": "DESVIO_PADRAO",
        "min": "MINIMO",
        "max": "MAXIMO"
    })
)


# ============================================================
# 5. Contagens e métricas agregadas dos testes externos
# ============================================================

contagens = [
    "CLIENTES_TESTE",
    "CLASSE_1_TESTE",
    "VERDADEIROS_NEGATIVOS",
    "FALSOS_POSITIVOS",
    "FALSOS_NEGATIVOS",
    "VERDADEIROS_POSITIVOS",
    "CLIENTES_SINALIZADOS"
]

CONTAGENS_FBETAS = (
    RESULTADOS_FBETAS_FOLDS
    .groupby("ESTRATEGIA")[contagens]
    .sum()
    .reindex(ordem)
)

tp = CONTAGENS_FBETAS["VERDADEIROS_POSITIVOS"]
fp = CONTAGENS_FBETAS["FALSOS_POSITIVOS"]
fn = CONTAGENS_FBETAS["FALSOS_NEGATIVOS"]
sinalizados = CONTAGENS_FBETAS["CLIENTES_SINALIZADOS"]

# Estas métricas usam as contagens somadas.
# Podem diferir da média simples das métricas por fold.
RESUMO_OPERACIONAL_FBETAS = CONTAGENS_FBETAS[
    [
        "VERDADEIROS_POSITIVOS",
        "FALSOS_NEGATIVOS",
        "FALSOS_POSITIVOS",
        "CLIENTES_SINALIZADOS"
    ]
].copy()

RESUMO_OPERACIONAL_FBETAS["PRECISAO_AGREGADA"] = (
    tp.div(sinalizados.replace(0, np.nan)).fillna(0)
)

RESUMO_OPERACIONAL_FBETAS["RECALL_AGREGADO"] = (
    tp.div((tp + fn).replace(0, np.nan)).fillna(0)
)

RESUMO_OPERACIONAL_FBETAS["PERCENTUAL_ENCAMINHADOS"] = (
    100 * sinalizados / CONTAGENS_FBETAS["CLIENTES_TESTE"]
)

RESUMO_OPERACIONAL_FBETAS[
    "ANALISES_POR_ATRASO_IDENTIFICADO"
] = sinalizados.div(tp.replace(0, np.nan))


# ============================================================
# 6. Apresentar resultados
# ============================================================

print("\nLIMIARES ESCOLHIDOS NOS FOLDS INTERNOS:")
display(LIMIARES_FBETAS_FOLDS.round(4))

print("\nMÉDIAS DOS CINCO TESTES EXTERNOS:")
display(MEDIAS_FBETAS.round(4))

print("\nDESVIOS PADRÃO DOS CINCO TESTES EXTERNOS:")
display(DESVIOS_FBETAS.round(4))

print("\nVARIAÇÃO DOS LIMIARES:")
display(RESUMO_LIMIARES_FBETAS.round(4))

print("\nRESULTADO OPERACIONAL AGREGADO:")
display(RESUMO_OPERACIONAL_FBETAS.round(4))

print("\nDIFERENÇAS EM RELAÇÃO AO LIMIAR PADRÃO:")
display(
    RESUMO_DIFERENCAS_FBETAS.loc[
        (
            slice(None),
            [
                "PRECISAO_CLASSE_1",
                "RECALL_CLASSE_1",
                "F1_CLASSE_1",
                "F2_CLASSE_1",
                "F3_CLASSE_1",
                "PERCENTUAL_ENCAMINHADOS"
            ]
        ),
        :
    ].round(4)
)

for estrategia in ordem:
    linha = CONTAGENS_FBETAS.loc[estrategia]

    matriz = pd.DataFrame(
        [
            [
                int(linha["VERDADEIROS_NEGATIVOS"]),
                int(linha["FALSOS_POSITIVOS"])
            ],
            [
                int(linha["FALSOS_NEGATIVOS"]),
                int(linha["VERDADEIROS_POSITIVOS"])
            ]
        ],
        index=["Real: classe 0", "Real: classe 1"],
        columns=["Previsto: classe 0", "Previsto: classe 1"]
    )

    print(f"\nMATRIZ DE CONFUSÃO — {estrategia}:")
    display(matriz)

Clientes: 21,731
Classe 1: 370
Folds externos: 5

Fold externo 1/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar por F1: 0.7365
  Limiar por F2: 0.3979
  Limiar por F3: 0.0272

Fold externo 2/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar por F1: 0.7502
  Limiar por F2: 0.3138
  Limiar por F3: 0.0724

Fold externo 3/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar por F1: 0.6170
  Limiar por F2: 0.5739
  Limiar por F3: 0.0696

Fold externo 4/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar por F1: 0.6908
  Limiar por F2: 0.0372
  Limiar por F3: 0.0080

Fold externo 5/5
  Treinando fold interno 1/3
  Treinando fold interno 2/3
  Treinando fold interno 3/3
  Limiar por F1: 0.7283
  Limiar por F2: 0.4364
  Limiar por F3: 0.1363

LIMIARES ESCOLHIDOS NOS FOLDS INTERNOS:


    FOLD  CLIENTES_TREINO  CLIENTES_TESTE  CLASSE_1_TREINO  CLASSE_1_TESTE  \
0      1            17455            4276              300              70   
1      1            17455            4276              300              70   
2      1            17455            4276              300              70   
3      2            17483            4248              276              94   
4      2            17483            4248              276              94   
5      2            17483            4248              276              94   
6      3            17222            4509              292              78   
7      3            17222            4509              292              78   
8      3            17222            4509              292              78   
9      4            17495            4236              304              66   
10     4            17495            4236              304              66   
11     4            17495            4236              304      


MÉDIAS DOS CINCO TESTES EXTERNOS:


               ACURACIA  ACURACIA_BALANCEADA  PRECISAO_CLASSE_1  \
ESTRATEGIA                                                        
Limiar padrão    0.9613               0.5856             0.1185   
Limiar por F1    0.9774               0.5642             0.2273   
Limiar por F2    0.8609               0.5713             0.0758   
Limiar por F3    0.5934               0.5678             0.0252   

               RECALL_CLASSE_1  F1_CLASSE_1  F2_CLASSE_1  F3_CLASSE_1  \
ESTRATEGIA                                                              
Limiar padrão           0.1967       0.1455       0.1715       0.1830   
Limiar por F1           0.1364       0.1661       0.1464       0.1412   
Limiar por F2           0.2712       0.0962       0.1314       0.1591   
Limiar por F3           0.5410       0.0475       0.1021       0.1669   

               AVERAGE_PRECISION  ROC_AUC  PERCENTUAL_ENCAMINHADOS  
ESTRATEGIA                                                          
Limiar padrão       


DESVIOS PADRÃO DOS CINCO TESTES EXTERNOS:


...

                Previsto: classe 0  Previsto: classe 1
Real: classe 0               21190                 171
Real: classe 1                 320                  50


MATRIZ DE CONFUSÃO — Limiar por F2:


                Previsto: classe 0  Previsto: classe 1
Real: classe 0               18668                2693
Real: classe 1                 273                  97


MATRIZ DE CONFUSÃO — Limiar por F3:


                Previsto: classe 0  Previsto: classe 1
Real: classe 0               12776                8585
Real: classe 1                 173                 197

**Leitura (resultados do Colab, 5 folds):**

| Estratégia | Precisão | Recall | Clientes encaminhados |
|---|---|---|---|
| Limiar padrão (0,50) | 11,9% | 19,7% | 2,8% |
| Limiar por F1 | 22,7% | 13,6% | 1,0% |
| Limiar por F2 | 7,6% | 27,1% | 13,1% |
| Limiar por F3 | 2,5% | 54,1% | 40,8% |

Os limiares variam muito entre folds (F2: 0,04 a 0,57), o que mostra instabilidade. O limiar 0,50 foi mantido como equilíbrio simples entre os dois erros.

## 3. Importância das variáveis (permutação)

Queda média de Average Precision ao embaralhar cada variável no teste externo, em cada um dos 5 folds. Valores negativos ou instáveis indicam variáveis sem contribuição consistente. A importância mostra o quanto o modelo usa a variável, não o sentido do efeito.

In [ ]:
#AVALIA A IMPORTANCIA DAS VARIAVEIS POR PERMUTAÇÃO E SUA CONSISTENCIA ENTRE FOLDS
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

# ============================================================
# 1. Configuração
# ============================================================

necessarios = [
    "X_limiar",
    "y_limiar",
    "grupos_limiar",
    "modelo_base",
    "plano_folds",
    "validar_divisao",
    "probabilidades_classe_1"
]

faltantes = [
    nome for nome in necessarios
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        "Execute primeiro as células anteriores. "
        f"Objetos ausentes: {faltantes}"
    )

SEMENTE_PERMUTACAO = 42
REPETICOES = 10

if not X_limiar.columns.is_unique:
    raise ValueError("Existem nomes de colunas repetidos.")

if {"ID", "ALVO"}.intersection(X_limiar.columns):
    raise ValueError("ID e ALVO não podem entrar como preditores.")

modelo_referencia = clone(modelo_base)
modelo_referencia.set_params(
    modelo__early_stopping=False
)

print(f"Clientes: {len(y_limiar):,}")
print(f"Variáveis: {X_limiar.shape[1]}")
print(f"Folds externos: {len(plano_folds)}")
print(f"Permutações por variável e fold: {REPETICOES}")
print("Métrica: queda de Average Precision")


# ============================================================
# 2. Importância por permutação no teste externo
# ============================================================

registros_folds = []
registros_repeticoes = []
registros_diagnostico = []

for fold, treino, teste, _ in plano_folds:
    print(f"\nFold externo {fold}/{len(plano_folds)}")

    validar_divisao(
        y_limiar,
        grupos_limiar,
        treino,
        teste,
        f"Fold externo {fold}"
    )

    X_treino = X_limiar.iloc[treino]
    y_treino = y_limiar.iloc[treino]

    X_teste = X_limiar.iloc[teste]
    y_teste = y_limiar.iloc[teste]

    # Imputação, codificação e modelo aprendem só no treino.
    modelo = clone(modelo_referencia)
    modelo.fit(X_treino, y_treino)

    probabilidades = probabilidades_classe_1(
        modelo, X_teste
    )

    ap_original = average_precision_score(
        y_teste, probabilidades
    )

    print(f"  AP original: {ap_original:.4f}")
    print("  Calculando permutações...")

    resultado = permutation_importance(
        estimator=modelo,
        X=X_teste,
        y=y_teste,
        scoring="average_precision",
        n_repeats=REPETICOES,
        random_state=SEMENTE_PERMUTACAO + fold,
        n_jobs=1,
        max_samples=1.0
    )

    # Ranking dentro do fold, calculado pela importância média.
    rankings = pd.Series(
        resultado.importances_mean,
        index=X_limiar.columns
    ).rank(ascending=False, method="min")

    for indice, variavel in enumerate(X_limiar.columns):
        quedas = resultado.importances[indice]

        registros_folds.append({
            "FOLD": fold,
            "VARIAVEL": variavel,
            "AP_ORIGINAL": ap_original,
            "IMPORTANCIA_MEDIA": float(quedas.mean()),
            "DESVIO_PERMUTACOES": float(
                quedas.std(ddof=1)
            ),
            "QUEDA_MINIMA": float(quedas.min()),
            "QUEDA_MAXIMA": float(quedas.max()),
            "PERCENTUAL_PERMUTACOES_POSITIVAS": (
                100 * np.mean(quedas > 0)
            ),
            "POSICAO_NO_FOLD": int(rankings.loc[variavel])
        })

        for repeticao, queda in enumerate(quedas, start=1):
            registros_repeticoes.append({
                "FOLD": fold,
                "VARIAVEL": variavel,
                "REPETICAO": repeticao,
                "AP_ORIGINAL": ap_original,
                "AP_PERMUTADA": ap_original - float(queda),
                "QUEDA_AP": float(queda)
            })

    registros_diagnostico.append({
        "FOLD": fold,
        "CLIENTES_TREINO": len(treino),
        "CLIENTES_TESTE": len(teste),
        "CLASSE_1_TESTE": int(y_teste.sum()),
        "PERFIS_COMPARTILHADOS": 0,
        "AVERAGE_PRECISION_ORIGINAL": ap_original
    })


# ============================================================
# 3. Consolidar a consistência entre folds
# ============================================================

IMPORTANCIAS_PERMUTACAO_FOLDS = pd.DataFrame(
    registros_folds
)

IMPORTANCIAS_PERMUTACAO_REPETICOES = pd.DataFrame(
    registros_repeticoes
)

DIAGNOSTICO_PERMUTACAO = pd.DataFrame(
    registros_diagnostico
)

RESUMO_IMPORTANCIAS = (
    IMPORTANCIAS_PERMUTACAO_FOLDS
    .groupby("VARIAVEL")
    .agg(
        IMPORTANCIA_MEDIA=(
            "IMPORTANCIA_MEDIA", "mean"
        ),
        DESVIO_ENTRE_FOLDS=(
            "IMPORTANCIA_MEDIA", "std"
        ),
        MENOR_IMPORTANCIA_FOLD=(
            "IMPORTANCIA_MEDIA", "min"
        ),
        MAIOR_IMPORTANCIA_FOLD=(
            "IMPORTANCIA_MEDIA", "max"
        ),
        FOLDS_IMPORTANCIA_POSITIVA=(
            "IMPORTANCIA_MEDIA",
            lambda s: int(s.gt(0).sum())
        ),
        FOLDS_NO_TOP_5=(
            "POSICAO_NO_FOLD",
            lambda s: int(s.le(5).sum())
        ),
        POSICAO_MEDIA=(
            "POSICAO_NO_FOLD", "mean"
        ),
        DESVIO_POSICAO=(
            "POSICAO_NO_FOLD", "std"
        ),
        DESVIO_MEDIO_PERMUTACOES=(
            "DESVIO_PERMUTACOES", "mean"
        )
    )
    .sort_values("IMPORTANCIA_MEDIA", ascending=False)
)

# Mantém valores negativos: não transforma em zero.
IMPORTANCIAS_POR_FOLD = (
    IMPORTANCIAS_PERMUTACAO_FOLDS
    .pivot(
        index="VARIAVEL",
        columns="FOLD",
        values="IMPORTANCIA_MEDIA"
    )
    .reindex(RESUMO_IMPORTANCIAS.index)
)

POSICOES_POR_FOLD = (
    IMPORTANCIAS_PERMUTACAO_FOLDS
    .pivot(
        index="VARIAVEL",
        columns="FOLD",
        values="POSICAO_NO_FOLD"
    )
    .reindex(RESUMO_IMPORTANCIAS.index)
)

# Diagnóstico de redundância exata ou colunas constantes.
# Serve apenas à interpretação; nenhuma coluna é removida.
variaveis_constantes = [
    coluna for coluna in X_limiar.columns
    if X_limiar[coluna].nunique(dropna=False) <= 1
]

pares_identicos = []

colunas = list(X_limiar.columns)

for i, coluna_a in enumerate(colunas):
    for coluna_b in colunas[i + 1:]:
        a = X_limiar[coluna_a]
        b = X_limiar[coluna_b]

        iguais = (
            a.eq(b).fillna(False)
            | (a.isna() & b.isna())
        )

        if iguais.all():
            pares_identicos.append({
                "VARIAVEL_A": coluna_a,
                "VARIAVEL_B": coluna_b
            })

COLUNAS_IDENTICAS = pd.DataFrame(
    pares_identicos,
    columns=["VARIAVEL_A", "VARIAVEL_B"]
)


# ============================================================
# 4. Apresentar resultados
# ============================================================

print("\nDESEMPENHO ORIGINAL EM CADA TESTE EXTERNO:")
display(DIAGNOSTICO_PERMUTACAO.round(4))

print("\nIMPORTÂNCIA E CONSISTÊNCIA ENTRE FOLDS:")
display(RESUMO_IMPORTANCIAS.round(5))

print("\nQUEDA MÉDIA DE AP POR VARIÁVEL E FOLD:")
display(IMPORTANCIAS_POR_FOLD.round(5))

print("\nPOSIÇÃO DAS VARIÁVEIS EM CADA FOLD:")
display(POSICOES_POR_FOLD)

print("\nCOLUNAS CONSTANTES:")
print(variaveis_constantes or "Nenhuma.")

print("\nPARES DE COLUNAS COM VALORES IDÊNTICOS:")
display(COLUNAS_IDENTICAS)

Clientes: 21,731
Variáveis: 17
Folds externos: 5
Permutações por variável e fold: 10
Métrica: queda de Average Precision

Fold externo 1/5
  AP original: 0.1559
  Calculando permutações...

Fold externo 2/5
  AP original: 0.1164
  Calculando permutações...

Fold externo 3/5
  AP original: 0.0977
  Calculando permutações...

Fold externo 4/5
  AP original: 0.1839
  Calculando permutações...

Fold externo 5/5
  AP original: 0.0574
  Calculando permutações...

DESEMPENHO ORIGINAL EM CADA TESTE EXTERNO:


   FOLD  CLIENTES_TREINO  CLIENTES_TESTE  CLASSE_1_TESTE  \
0     1            17455            4276              70   
1     2            17483            4248              94   
2     3            17222            4509              78   
3     4            17495            4236              66   
4     5            17269            4462              62   

   PERFIS_COMPARTILHADOS  AVERAGE_PRECISION_ORIGINAL  
0                      0                      0.1559  
1                      0                      0.1164  
2                      0                      0.0977  
3                      0                      0.1839  
4                      0                      0.0574  


IMPORTÂNCIA E CONSISTÊNCIA ENTRE FOLDS:


                     IMPORTANCIA_MEDIA  DESVIO_ENTRE_FOLDS  \
VARIAVEL                                                     
DAYS_BIRTH                     0.08861             0.03981   
AMT_INCOME_TOTAL               0.07558             0.03581   
DAYS_EMPLOYED                  0.07062             0.04624   
OCCUPATION_TYPE                0.03981             0.03029   
NAME_FAMILY_STATUS             0.03812             0.03512   
NAME_INCOME_TYPE               0.02895             0.01496   
NAME_EDUCATION_TYPE            0.02062             0.01909   
CNT_FAM_MEMBERS                0.02002             0.01266   
FLAG_OWN_REALTY                0.01910             0.01735   
FLAG_PHONE                     0.01562             0.01271   
CODE_GENDER                    0.01111             0.01359   
FLAG_OWN_CAR                   0.00899             0.01210   
FLAG_WORK_PHONE                0.00712             0.00731   
NAME_HOUSING_TYPE              0.00649             0.00757   
CNT_CHIL


QUEDA MÉDIA DE AP POR VARIÁVEL E FOLD:


FOLD                       1        2        3        4        5
VARIAVEL                                                        
DAYS_BIRTH           0.12051  0.08685  0.06680  0.13349  0.03543
AMT_INCOME_TOTAL     0.10913  0.07358  0.05144  0.11305  0.03070
DAYS_EMPLOYED        0.09364  0.07586  0.03907  0.13140  0.01312
OCCUPATION_TYPE      0.05243  0.04551  0.02637  0.07791 -0.00317
NAME_FAMILY_STATUS   0.02538  0.03074  0.03180  0.09786  0.00484
NAME_INCOME_TYPE     0.04490  0.01453  0.02382  0.04489  0.01659
NAME_EDUCATION_TYPE  0.02201  0.02519  0.00555  0.04932  0.00104
CNT_FAM_MEMBERS      0.03096  0.00176  0.01410  0.03232  0.02097
FLAG_OWN_REALTY      0.01295  0.01823  0.01466  0.04807  0.00157
FLAG_PHONE           0.03554  0.01658  0.00437  0.01695  0.00465
CODE_GENDER          0.02622 -0.00469  0.01166  0.02257 -0.00023
FLAG_OWN_CAR         0.01532  0.00092 -0.00294  0.02690  0.00475
FLAG_WORK_PHONE      0.00020  0.01696  0.00329  0.01276  0.00237
NAME_HOUSING_TYPE    0.00


POSIÇÃO DAS VARIÁVEIS EM CADA FOLD:


FOLD                  1   2   3   4   5
VARIAVEL                               
DAYS_BIRTH            1   1   1   1   1
AMT_INCOME_TOTAL      2   3   2   3   2
DAYS_EMPLOYED         3   2   3   2   5
OCCUPATION_TYPE       4   4   5   5  17
NAME_FAMILY_STATUS    9   5   4   4   7
NAME_INCOME_TYPE      5  10   6   8   4
NAME_EDUCATION_TYPE  10   6  13   6  12
CNT_FAM_MEMBERS       7  13   8   9   3
FLAG_OWN_REALTY      12   7   7   7  11
FLAG_PHONE            6   9  14  13   9
CODE_GENDER           8  16   9  11  15
FLAG_OWN_CAR         11  14  17  10   8
FLAG_WORK_PHONE      14   8  15  14  10
NAME_HOUSING_TYPE    13  11  10  12  16
CNT_CHILDREN         16  12  11  15   6
FLAG_MOBIL           15  15  16  17  13
FLAG_EMAIL           17  17  12  16  14


COLUNAS CONSTANTES:
['FLAG_MOBIL']

PARES DE COLUNAS COM VALORES IDÊNTICOS:


Empty DataFrame
Columns: [VARIAVEL_A, VARIAVEL_B]
Index: []

**Leitura (resultados do Colab):** `DAYS_BIRTH` (idade) é a mais importante nos 5 folds; `AMT_INCOME_TOTAL` (renda) e `DAYS_EMPLOYED` (tempo de emprego) vêm em seguida, quase sempre entre as 3 primeiras. `OCCUPATION_TYPE`, `NAME_FAMILY_STATUS` e `NAME_INCOME_TYPE` ficam num segundo bloco, e `FLAG_EMAIL`, `FLAG_WORK_PHONE`, `CNT_CHILDREN` e `NAME_HOUSING_TYPE` contribuem pouco. `FLAG_MOBIL` é constante.

## 4. Limites do sinal cadastral

Clientes com cadastro idêntico podem ter desfechos opostos. Dos 8.744 perfis distintos, 248 (2,8%) têm clientes bons e ruins ao mesmo tempo, e 63% dos inadimplentes estão nesses perfis. Dentro deles, quem atrasa tem histórico mais longo, o que indica que o cadastro sozinho não separa os dois grupos.

In [ ]:
#VERIFICA PERFIS CADASTRAIS IGUAIS COM ALVOS DIFERENTES

import pandas as pd
from IPython.display import display

# 1. Preparar uma cópia.
base_auditoria = BASE_MODELO.copy().reset_index(drop=True)

if base_auditoria['ID'].isna().any():
    raise ValueError('Existem IDs ausentes.')

if base_auditoria['ID'].duplicated().any():
    raise ValueError('A base deve conter uma única linha por ID.')

if (
    base_auditoria['ALVO'].isna().any()
    or not base_auditoria['ALVO'].isin([0, 1]).all()
):
    raise ValueError('ALVO deve conter somente 0 e 1, sem ausências.')

colunas_perfil = [
    coluna for coluna in base_auditoria.columns
    if coluna not in ['ID', 'ALVO']
]

# 2. Identificar perfis iguais, incluindo ausências iguais.
grupos = (
    base_auditoria.groupby(
        colunas_perfil,
        dropna=False,
        observed=True,
        sort=False
    )
    .ngroup()
)

base_auditoria['GRUPO_PERFIL'] = grupos

# 3. Contar clientes de cada classe por perfil.
RESUMO_ALVOS_POR_PERFIL = (
    base_auditoria.groupby('GRUPO_PERFIL')
    .agg(
        CLIENTES=('ID', 'nunique'),
        CLASSE_1=('ALVO', 'sum'),
        QUANTIDADE_CLASSES=('ALVO', 'nunique')
    )
)

RESUMO_ALVOS_POR_PERFIL['CLASSE_0'] = (
    RESUMO_ALVOS_POR_PERFIL['CLIENTES']
    - RESUMO_ALVOS_POR_PERFIL['CLASSE_1']
)

RESUMO_ALVOS_POR_PERFIL['PERCENTUAL_CLASSE_1'] = (
    100
    * RESUMO_ALVOS_POR_PERFIL['CLASSE_1']
    / RESUMO_ALVOS_POR_PERFIL['CLIENTES']
).round(2)

RESUMO_ALVOS_POR_PERFIL['ALVOS_DIFERENTES'] = (
    RESUMO_ALVOS_POR_PERFIL['QUANTIDADE_CLASSES'].eq(2)
)

# 4. Separar os perfis com ambas as classes.
PERFIS_ALVOS_DIFERENTES = (
    RESUMO_ALVOS_POR_PERFIL.loc[
        RESUMO_ALVOS_POR_PERFIL['ALVOS_DIFERENTES']
    ]
    .sort_values('CLIENTES', ascending=False)
    .copy()
)

CLIENTES_ALVOS_DIFERENTES = (
    base_auditoria.loc[
        base_auditoria['GRUPO_PERFIL']
        .isin(PERFIS_ALVOS_DIFERENTES.index)
    ]
    .sort_values(['GRUPO_PERFIL', 'ALVO', 'ID'])
    .reset_index(drop=True)
)

# 5. Diagnóstico geral.
total_perfis = len(RESUMO_ALVOS_POR_PERFIL)
total_clientes = len(base_auditoria)
total_classe_1 = int(base_auditoria['ALVO'].sum())

perfis_mistos = len(PERFIS_ALVOS_DIFERENTES)
clientes_mistos = len(CLIENTES_ALVOS_DIFERENTES)
classe_1_mistos = int(PERFIS_ALVOS_DIFERENTES['CLASSE_1'].sum())

def percentual(parte, total):
    return round(100 * parte / total, 2) if total else float('nan')

DIAGNOSTICO_ALVOS_POR_PERFIL = pd.DataFrame([{
    'TOTAL_PERFIS': total_perfis,
    'PERFIS_COM_ALVOS_DIFERENTES': perfis_mistos,
    'PERCENTUAL_PERFIS_MISTOS': percentual(
        perfis_mistos, total_perfis
    ),
    'TOTAL_CLIENTES': total_clientes,
    'CLIENTES_EM_PERFIS_MISTOS': clientes_mistos,
    'PERCENTUAL_CLIENTES_EM_PERFIS_MISTOS': percentual(
        clientes_mistos, total_clientes
    ),
    'TOTAL_CLIENTES_CLASSE_1': total_classe_1,
    'CLASSE_1_EM_PERFIS_MISTOS': classe_1_mistos,
    'PERCENTUAL_CLASSE_1_EM_PERFIS_MISTOS': percentual(
        classe_1_mistos, total_classe_1
    )
}])

print('DIAGNÓSTICO GERAL:')
display(DIAGNOSTICO_ALVOS_POR_PERFIL.T.rename(
    columns={0: 'RESULTADO'}
)),

DIAGNÓSTICO GERAL:


                                      RESULTADO
TOTAL_PERFIS                            8744.00
PERFIS_COM_ALVOS_DIFERENTES              248.00
PERCENTUAL_PERFIS_MISTOS                   2.84
TOTAL_CLIENTES                         29029.00
CLIENTES_EM_PERFIS_MISTOS               1291.00
PERCENTUAL_CLIENTES_EM_PERFIS_MISTOS       4.45
TOTAL_CLIENTES_CLASSE_1                  608.00
CLASSE_1_EM_PERFIS_MISTOS                383.00
PERCENTUAL_CLASSE_1_EM_PERFIS_MISTOS      62.99

(None,)

In [ ]:
#COMPARA DURAÇÃO DO HISTORICO E MESES COM CREDITO OBSERVADO ENTRE AS CLASSES

import pandas as pd
from IPython.display import display

# 1. Preparar o histórico.
historico = CREDIT_FILTRADO[
    ['ID', 'MONTHS_BALANCE', 'STATUS']
].copy()

historico['ID'] = historico['ID'].astype('string').str.strip()
historico['MONTHS_BALANCE'] = pd.to_numeric(
    historico['MONTHS_BALANCE'], errors='raise'
)
historico['STATUS'] = (
    historico['STATUS'].astype('string').str.strip()
)

if historico['ID'].isna().any() or historico['ID'].eq('').any():
    raise ValueError('Existem IDs ausentes ou vazios.')

if (
    historico['MONTHS_BALANCE'].isna().any()
    or historico['MONTHS_BALANCE'].mod(1).ne(0).any()
    or historico['MONTHS_BALANCE'].gt(0).any()
):
    raise ValueError('Existem meses ausentes ou inválidos.')

if not historico['STATUS'].isin(
    ['0', '1', '2', '3', '4', '5', 'C', 'X']
).all():
    raise ValueError('Existem STATUS ausentes ou desconhecidos.')

if historico.duplicated(['ID', 'MONTHS_BALANCE']).any():
    raise ValueError('Existem registros duplicados de ID + mês.')

historico['DIFERENTE_X'] = historico['STATUS'].ne('X')
historico['STATUS_NUMERICO'] = historico['STATUS'].isin(
    ['0', '1', '2', '3', '4', '5']
)

# 2. Resumir a duração e a exposição por cliente.
RESUMO_DURACAO_CREDITO = (
    historico.groupby('ID')
    .agg(
        PRIMEIRO_MES=('MONTHS_BALANCE', 'min'),
        ULTIMO_MES=('MONTHS_BALANCE', 'max'),
        MESES_REGISTRADOS=('MONTHS_BALANCE', 'nunique'),
        MESES_DIFERENTES_X=('DIFERENTE_X', 'sum'),
        MESES_STATUS_NUMERICO=('STATUS_NUMERICO', 'sum')
    )
    .reset_index()
)

RESUMO_DURACAO_CREDITO['INTERVALO_MESES'] = (
    RESUMO_DURACAO_CREDITO['ULTIMO_MES']
    - RESUMO_DURACAO_CREDITO['PRIMEIRO_MES']
    + 1
)

RESUMO_DURACAO_CREDITO['LACUNAS'] = (
    RESUMO_DURACAO_CREDITO['INTERVALO_MESES']
    - RESUMO_DURACAO_CREDITO['MESES_REGISTRADOS']
)

# 3. Incorporar o alvo e o grupo cadastral já definidos.
clientes = base_auditoria[
    ['ID', 'ALVO', 'GRUPO_PERFIL']
].copy()

clientes['ID'] = clientes['ID'].astype('string').str.strip()

DIAGNOSTICO_DURACAO_CLIENTES = clientes.merge(
    RESUMO_DURACAO_CREDITO,
    on='ID',
    how='left',
    validate='one_to_one',
    indicator=True
)

if DIAGNOSTICO_DURACAO_CLIENTES['_merge'].ne('both').any():
    raise ValueError(
        'Há clientes da base do modelo sem histórico correspondente. '
        'Confira a tabela de crédito e o formato dos IDs.'
    )

DIAGNOSTICO_DURACAO_CLIENTES = (
    DIAGNOSTICO_DURACAO_CLIENTES.drop(columns='_merge')
)

# Identificar perfis com ambas as classes.
classes_por_perfil = (
    clientes.groupby('GRUPO_PERFIL')['ALVO'].nunique()
)

ids_perfis_mistos = classes_por_perfil[
    classes_por_perfil.eq(2)
].index

DIAGNOSTICO_DURACAO_CLIENTES['PERFIL_MISTO'] = (
    DIAGNOSTICO_DURACAO_CLIENTES['GRUPO_PERFIL']
    .isin(ids_perfis_mistos)
)

clientes_mistos = DIAGNOSTICO_DURACAO_CLIENTES.loc[
    DIAGNOSTICO_DURACAO_CLIENTES['PERFIL_MISTO']
].copy()

# 4. Comparar as distribuições por classe.
medidas = [
    'MESES_REGISTRADOS',
    'MESES_DIFERENTES_X',
    'MESES_STATUS_NUMERICO'
]

def resumir_por_classe(tabela, nome_escopo):
    linhas = []

    for classe, grupo in tabela.groupby('ALVO'):
        for medida in medidas:
            valores = grupo[medida]

            linhas.append({
                'ESCOPO': nome_escopo,
                'CLASSE': classe,
                'MEDIDA': medida,
                'CLIENTES': len(grupo),
                'MEDIA': valores.mean(),
                'MEDIANA': valores.median(),
                'P25': valores.quantile(0.25),
                'P75': valores.quantile(0.75),
                'MINIMO': valores.min(),
                'MAXIMO': valores.max()
            })

    return pd.DataFrame(linhas)

COMPARACAO_DURACAO_CLASSES = pd.concat(
    [
        resumir_por_classe(
            DIAGNOSTICO_DURACAO_CLIENTES, 'Base completa'
        ),
        resumir_por_classe(
            clientes_mistos, 'Somente perfis mistos'
        )
    ],
    ignore_index=True
)

print('DURAÇÃO E EXPOSIÇÃO POR CLASSE:')
display(COMPARACAO_DURACAO_CLASSES.round(2))

# 5. Comparação dentro de cada perfil misto.
# Diferença = média da classe 1 menos média da classe 0.
# Cada perfil terá o mesmo peso no resumo abaixo.
diferencas = []
resumo_diferencas = []

if not clientes_mistos.empty:
    medias_por_perfil = (
        clientes_mistos.groupby(['GRUPO_PERFIL', 'ALVO'])[medidas]
        .mean()
    )

    for medida in medidas:
        medias = medias_por_perfil[medida].unstack('ALVO')
        delta = medias[1] - medias[0]

        diferencas.append(pd.DataFrame({
            'GRUPO_PERFIL': delta.index,
            'MEDIDA': medida,
            'MEDIA_CLASSE_0': medias[0].values,
            'MEDIA_CLASSE_1': medias[1].values,
            'DIFERENCA_1_MENOS_0': delta.values
        }))

        # Evitar diferenças residuais de ponto flutuante nos empates.
        sinal = delta.round(8)

        resumo_diferencas.append({
            'MEDIDA': medida,
            'PERFIS_COMPARADOS': len(delta),
            'DIFERENCA_MEDIA': delta.mean(),
            'DIFERENCA_MEDIANA': delta.median(),
            'PCT_PERFIS_CLASSE_1_MAIOR': 100 * sinal.gt(0).mean(),
            'PCT_PERFIS_IGUAIS': 100 * sinal.eq(0).mean(),
            'PCT_PERFIS_CLASSE_1_MENOR': 100 * sinal.lt(0).mean()
        })

    DIFERENCAS_DENTRO_PERFIL = pd.concat(
        diferencas, ignore_index=True
    )

    RESUMO_DIFERENCAS_PERFIS = pd.DataFrame(resumo_diferencas)

    print('\nCOMPARAÇÃO DENTRO DOS PERFIS MISTOS:')
    display(RESUMO_DIFERENCAS_PERFIS.round(2))
else:
    DIFERENCAS_DENTRO_PERFIL = pd.DataFrame()
    RESUMO_DIFERENCAS_PERFIS = pd.DataFrame()
    print('\nNão existem perfis mistos para comparar.')

print(
    '\nClientes com lacunas no histórico: '
    f"{DIAGNOSTICO_DURACAO_CLIENTES['LACUNAS'].gt(0).sum():,}"
)

DURAÇÃO E EXPOSIÇÃO POR CLASSE:


                   ESCOPO  CLASSE                 MEDIDA  CLIENTES  MEDIA  \
0           Base completa       0      MESES_REGISTRADOS     28421  23.79   
1           Base completa       0     MESES_DIFERENTES_X     28421  21.19   
2           Base completa       0  MESES_STATUS_NUMERICO     28421   9.89   
3           Base completa       1      MESES_REGISTRADOS       608  33.76   
4           Base completa       1     MESES_DIFERENTES_X       608  29.65   
5           Base completa       1  MESES_STATUS_NUMERICO       608  17.95   
6   Somente perfis mistos       0      MESES_REGISTRADOS       908  25.06   
7   Somente perfis mistos       0     MESES_DIFERENTES_X       908  22.15   
8   Somente perfis mistos       0  MESES_STATUS_NUMERICO       908  10.12   
9   Somente perfis mistos       1      MESES_REGISTRADOS       383  34.78   
10  Somente perfis mistos       1     MESES_DIFERENTES_X       383  30.38   
11  Somente perfis mistos       1  MESES_STATUS_NUMERICO       383  18.92   


COMPARAÇÃO DENTRO DOS PERFIS MISTOS:


                  MEDIDA  PERFIS_COMPARADOS  DIFERENCA_MEDIA  \
0      MESES_REGISTRADOS                248            10.31   
1     MESES_DIFERENTES_X                248             8.58   
2  MESES_STATUS_NUMERICO                248             8.58   

   DIFERENCA_MEDIANA  PCT_PERFIS_CLASSE_1_MAIOR  PCT_PERFIS_IGUAIS  \
0               9.67                      75.40               0.40   
1               7.88                      68.95               0.81   
2               6.50                      72.98               2.02   

   PCT_PERFIS_CLASSE_1_MENOR  
0                      24.19  
1                      30.24  
2                      25.00  


Clientes com lacunas no histórico: 0


## 5. Modelo final

Hist Gradient Boosting, janela de 12 meses, limiar 0,50, treinado nos 21.731 clientes elegíveis com 16 preditores (`FLAG_MOBIL`, constante, é removida). O artefato salvo contém o pipeline, os metadados (definição do alvo, limitações, versões) e a função `classificar_cadastros`. Os resultados de validação (seção 2) são os únicos indicadores de desempenho; o exemplo ao final desta célula não é uma avaliação.

In [ ]:
#TREINA O MODELO ATUAL

import os
import platform
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone

# ============================================================
# 1. Configuração
# ============================================================

necessarios = [
    "X_limiar",
    "y_limiar",
    "ids_limiar",
    "modelo_base"
]

faltantes = [
    nome for nome in necessarios
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Execute as células anteriores. Objetos ausentes: {faltantes}"
    )

LIMIAR_FINAL = 0.50

CAMINHO_MODELO = PROCESSED / "modelo_hgb_final_12_meses.joblib"

X_final = X_limiar.copy()
y_final = y_limiar.copy()

if not X_final.columns.is_unique:
    raise ValueError("Existem nomes de colunas duplicados.")

if len(X_final) != len(y_final):
    raise ValueError("X e y possuem quantidades diferentes de linhas.")

if set(y_final.unique()) != {0, 1}:
    raise ValueError("O alvo precisa conter somente as classes 0 e 1.")

if {"ID", "ALVO"}.intersection(X_final.columns):
    raise ValueError("ID e ALVO não podem ser preditores.")

# Mantém os parâmetros do HGB atual.
modelo_final = clone(modelo_base)
modelo_final.set_params(modelo__early_stopping=False)

# ============================================================
# 2. Retirar somente FLAG_MOBIL, se ela for constante
# ============================================================

colunas_removidas = []

if "FLAG_MOBIL" in X_final.columns:
    if X_final["FLAG_MOBIL"].nunique(dropna=False) > 1:
        raise ValueError(
            "FLAG_MOBIL deixou de ser constante. "
            "Revise a base antes de continuar."
        )

    # Localiza o ColumnTransformer do pipeline.
    preprocessadores = [
        (nome, etapa)
        for nome, etapa in modelo_final.steps
        if hasattr(etapa, "transformers")
    ]

    if len(preprocessadores) != 1:
        raise ValueError(
            "Não foi possível identificar um único "
            "ColumnTransformer no pipeline."
        )

    nome_preprocessador, preprocessador = preprocessadores[0]

    novos_transformadores = []

    for nome, transformador, colunas in preprocessador.transformers:
        # O pipeline anterior usa listas de nomes de colunas.
        if not isinstance(colunas, (list, tuple, pd.Index, np.ndarray)):
            raise ValueError(
                "O pré-processador usa um seletor de colunas "
                "não previsto neste código."
            )

        if not all(isinstance(c, str) for c in colunas):
            raise ValueError(
                "Os seletores precisam conter nomes de colunas."
            )

        novas_colunas = [
            c for c in colunas if c != "FLAG_MOBIL"
        ]

        if novas_colunas:
            novos_transformadores.append(
                (nome, transformador, novas_colunas)
            )

    modelo_final.set_params(**{
        f"{nome_preprocessador}__transformers":
            novos_transformadores
    })

    X_final = X_final.drop(columns="FLAG_MOBIL")
    colunas_removidas = ["FLAG_MOBIL"]

# ============================================================
# 3. Registrar o esquema e treinar em toda a base
# ============================================================

colunas_numericas = (
    X_final.select_dtypes(include="number").columns.tolist()
)

colunas_categoricas = [
    c for c in X_final.columns
    if c not in colunas_numericas
]

# Mesma padronização utilizada nos experimentos.
for coluna in colunas_numericas:
    X_final[coluna] = pd.to_numeric(
        X_final[coluna], errors="raise"
    ).astype("float64")

    if np.isinf(X_final[coluna].to_numpy()).any():
        raise ValueError(f"Existem infinitos em {coluna}.")

for coluna in colunas_categoricas:
    valores = X_final[coluna].astype("string")
    X_final[coluna] = (
        valores.astype(object)
        .where(valores.notna(), np.nan)
    )

print(f"Treinando em {len(X_final):,} clientes...")
modelo_final.fit(X_final, y_final)

# ============================================================
# 4. Metadados e resultados históricos de validação
# ============================================================

# Guarda resultados já calculados; não mede desempenho no treino.
resultados_validacao = {}

if "RESULTADOS_BUSCA_HGB_FOLDS" in globals():
    resultados_validacao["hgb_atual_por_fold"] = (
        RESULTADOS_BUSCA_HGB_FOLDS.loc[
            RESULTADOS_BUSCA_HGB_FOLDS["MODELO"].eq("HGB atual")
        ].copy()
    )

metadados = {
    "criado_em_utc": datetime.now(timezone.utc).isoformat(),
    "modelo": "HistGradientBoostingClassifier",
    "horizonte_meses": 12,
    "corte_atraso_dias": 60,
    "status_classe_1": ["2", "3", "4", "5"],
    "definicao_alvo": (
        "Ocorrência de atraso de 60+ dias nos primeiros "
        "12 meses disponíveis do histórico do cliente."
    ),
    "elegibilidade": (
        "Janela completa de 12 meses e pelo menos "
        "um mês com STATUS diferente de X."
    ),
    "limiar": LIMIAR_FINAL,
    "acao_classe_1": "Encaminhar para análise adicional",
    "natureza": "Classificação retrospectiva",
    "limitacao_temporal": (
        "Não foi confirmada a disponibilidade dos dados "
        "cadastrais antes da janela do alvo."
    ),
    "escores_calibrados": False,
    "validacao": (
        "Cinco folds externos separados por perfil cadastral. "
        "Resultados exploratórios."
    ),
    "clientes_treinamento": len(X_final),
    "clientes_classe_1": int(y_final.sum()),
    "percentual_classe_1": float(100 * y_final.mean()),
    "colunas_entrada": X_final.columns.tolist(),
    "colunas_numericas": colunas_numericas,
    "colunas_categoricas": colunas_categoricas,
    "colunas_removidas": colunas_removidas,
    "parametros_modelo": (
        modelo_final.named_steps["modelo"].get_params()
    ),
    "versoes": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "joblib": joblib.__version__
    }
}

ARTEFATO_MODELO_FINAL = {
    "pipeline": modelo_final,
    "metadados": metadados,
    "resultados_validacao": resultados_validacao
}

# ============================================================
# 5. Salvar um único arquivo em data/processed
# ============================================================

CAMINHO_MODELO.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    ARTEFATO_MODELO_FINAL,
    CAMINHO_MODELO,
    compress=3
)

# Confere se o arquivo pode ser carregado e se preserva os escores.
artefato_carregado = joblib.load(CAMINHO_MODELO)

amostra = X_final.head(10)

np.testing.assert_allclose(
    modelo_final.predict_proba(amostra),
    artefato_carregado["pipeline"].predict_proba(amostra)
)

print("\nModelo treinado e salvo.")
print(f"Arquivo: {CAMINHO_MODELO}")
print(f"Preditores: {X_final.shape[1]}")
print(f"Colunas retiradas: {colunas_removidas}")
print(f"Limiar: {LIMIAR_FINAL:.2f}")
print("Carregamento e preservação dos escores verificados.")


# ============================================================
# 6. Função para classificar novos cadastros já tratados
# ============================================================

def classificar_cadastros(cadastros, artefato):
    """
    Recebe um DataFrame de cadastros com o mesmo tratamento
    utilizado no treinamento. ID é opcional.
    """

    pipeline = artefato["pipeline"]
    metadata = artefato["metadados"]
    colunas = metadata["colunas_entrada"]

    ausentes = [
        c for c in colunas if c not in cadastros.columns
    ]

    if ausentes:
        raise ValueError(
            f"Colunas obrigatórias ausentes: {ausentes}"
        )

    if not cadastros.columns.is_unique:
        raise ValueError("Existem nomes de colunas duplicados.")

    if cadastros.empty:
        raise ValueError("Nenhum cadastro recebido.")

    entrada = cadastros[colunas].copy()

    for coluna in metadata["colunas_numericas"]:
        entrada[coluna] = pd.to_numeric(
            entrada[coluna], errors="raise"
        ).astype("float64")

        if np.isinf(entrada[coluna].to_numpy()).any():
            raise ValueError(f"Existem infinitos em {coluna}.")

    for coluna in metadata["colunas_categoricas"]:
        valores = entrada[coluna].astype("string")
        entrada[coluna] = (
            valores.astype(object)
            .where(valores.notna(), np.nan)
        )

    indice_classe_1 = list(pipeline.classes_).index(1)
    escores = pipeline.predict_proba(entrada)[:, indice_classe_1]

    sinalizacao = (
        escores >= metadata["limiar"]
    ).astype(int)

    resultado = pd.DataFrame({
        "ESCORE_CLASSE_1": escores,
        "SINALIZACAO": sinalizacao,
        "ACAO": np.where(
            sinalizacao == 1,
            "ENCAMINHAR_PARA_ANALISE",
            "SEM_SINALIZACAO_PELO_MODELO"
        )
    }, index=cadastros.index)

    if "ID" in cadastros.columns:
        resultado.insert(
            0, "ID", cadastros["ID"].to_numpy()
        )

    return resultado


# Exemplo de funcionamento em cinco cadastros do treinamento.
# Este exemplo NÃO é uma avaliação de desempenho.
exemplo = X_final.head(5).copy()
exemplo.insert(0, "ID", ids_limiar.iloc[:5].to_numpy())

display(
    classificar_cadastros(exemplo, artefato_carregado)
)

Treinando em 21,731 clientes...

Modelo treinado e salvo.
Arquivo: /content/drive/MyDrive/DataScienceFIAP/modelo_hgb_final_12_meses.joblib
Preditores: 16
Colunas retiradas: ['FLAG_MOBIL']
Limiar: 0.50
Carregamento e preservação dos escores verificados.


        ID  ESCORE_CLASSE_1  SINALIZACAO                         ACAO
0  5008804         0.066831            0  SEM_SINALIZACAO_PELO_MODELO
1  5008805         0.066831            0  SEM_SINALIZACAO_PELO_MODELO
2  5008806         0.029156            0  SEM_SINALIZACAO_PELO_MODELO
3  5008810         0.012367            0  SEM_SINALIZACAO_PELO_MODELO
4  5008811         0.012367            0  SEM_SINALIZACAO_PELO_MODELO

## 6. Aplicação à base cadastral inteira

Classifica os 438.510 cadastros únicos e lista quem seria encaminhado para análise. Esse número serve para dimensionar o esforço operacional, não como medida de desempenho: a base inclui os clientes usados no treino.

In [ ]:
#RODA PARA TODA A BASE DE IDs

import joblib
import numpy as np
import pandas as pd

from IPython.display import display

# 1. Carregar o modelo já treinado.
CAMINHO_MODELO = PROCESSED / "modelo_hgb_final_12_meses.joblib"

artefato = joblib.load(CAMINHO_MODELO)

pipeline = artefato["pipeline"]
metadados = artefato["metadados"]

colunas_entrada = metadados["colunas_entrada"]
limiar = metadados["limiar"]

# 2. Validar a base cadastral.
# base_total_trat: cadastro tratado, carregado na primeira célula.

cadastros = base_total_trat.copy()

if cadastros.empty:
    raise ValueError("A tabela base_total_trat está vazia.")

if not cadastros.columns.is_unique:
    raise ValueError("Existem nomes de colunas duplicados.")

if "ID" not in cadastros.columns:
    raise ValueError("A tabela precisa conter a coluna ID.")

ids_validacao = cadastros["ID"].astype("string").str.strip()

if ids_validacao.isna().any() or ids_validacao.eq("").any():
    raise ValueError("Existem IDs ausentes ou vazios.")

if ids_validacao.duplicated().any():
    raise ValueError(
        "Existem IDs duplicados. A base precisa conter "
        "uma única linha por cliente."
    )

ausentes = [
    coluna for coluna in colunas_entrada
    if coluna not in cadastros.columns
]

if ausentes:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {ausentes}"
    )

# Seleciona os preditores na ordem utilizada no treinamento.
entrada = cadastros[colunas_entrada].copy()

for coluna in metadados["colunas_numericas"]:
    entrada[coluna] = pd.to_numeric(
        entrada[coluna], errors="raise"
    ).astype("float64")

    if np.isinf(entrada[coluna].to_numpy()).any():
        raise ValueError(f"Existem valores infinitos em {coluna}.")

for coluna in metadados["colunas_categoricas"]:
    valores = entrada[coluna].astype("string")

    entrada[coluna] = (
        valores.astype(object)
        .where(valores.notna(), np.nan)
    )

# 3. Classificar em lotes, sem treinar novamente.
TAMANHO_LOTE = 10_000

indice_classe_1 = list(pipeline.classes_).index(1)
escores = np.empty(len(entrada), dtype=float)

for inicio in range(0, len(entrada), TAMANHO_LOTE):
    fim = min(inicio + TAMANHO_LOTE, len(entrada))

    escores[inicio:fim] = pipeline.predict_proba(
        entrada.iloc[inicio:fim]
    )[:, indice_classe_1]

if not np.isfinite(escores).all():
    raise ValueError("O modelo retornou escores inválidos.")

sinalizacao = (escores >= limiar).astype(int)

# Mantém todas as colunas cadastrais e adiciona os resultados.
RESULTADO_BASE_TOTAL_TRAT = cadastros.copy()

RESULTADO_BASE_TOTAL_TRAT["ESCORE_CLASSE_1"] = escores
RESULTADO_BASE_TOTAL_TRAT["LIMIAR_MODELO"] = limiar
RESULTADO_BASE_TOTAL_TRAT["SINALIZACAO"] = sinalizacao

RESULTADO_BASE_TOTAL_TRAT["ACAO"] = np.where(
    sinalizacao == 1,
    "ENCAMINHAR_PARA_ANALISE",
    "SEM_SINALIZACAO_PELO_MODELO"
)

# 4. Resumo e clientes sinalizados.
RESUMO_CLASSIFICACAO = (
    RESULTADO_BASE_TOTAL_TRAT
    .groupby(["SINALIZACAO", "ACAO"], dropna=False)
    .agg(CLIENTES=("ID", "nunique"))
    .reset_index()
)

RESUMO_CLASSIFICACAO["PERCENTUAL_CLIENTES"] = (
    100 * RESUMO_CLASSIFICACAO["CLIENTES"]
    / len(RESULTADO_BASE_TOTAL_TRAT)
)

CLIENTES_PARA_ANALISE = (
    RESULTADO_BASE_TOTAL_TRAT.loc[
        RESULTADO_BASE_TOTAL_TRAT["SINALIZACAO"].eq(1)
    ]
    .sort_values("ESCORE_CLASSE_1", ascending=False)
    .copy()
)

print(f"Clientes classificados: {len(RESULTADO_BASE_TOTAL_TRAT):,}")
print(f"Limiar utilizado: {limiar:.2f}")
print(f"Clientes sinalizados: {len(CLIENTES_PARA_ANALISE):,}")

display(RESUMO_CLASSIFICACAO.round(2))

print("\nExemplo dos resultados:")
display(
    RESULTADO_BASE_TOTAL_TRAT[
        ["ID", "ESCORE_CLASSE_1", "SINALIZACAO", "ACAO"]
    ].head(10)
)

print("\nClientes com maiores escores entre os sinalizados:")
display(
    CLIENTES_PARA_ANALISE[
        ["ID", "ESCORE_CLASSE_1", "SINALIZACAO", "ACAO"]
    ].head(20)
)

Clientes classificados: 438,510
Limiar utilizado: 0.50
Clientes sinalizados: 14,255


   SINALIZACAO                         ACAO  CLIENTES  PERCENTUAL_CLIENTES
0            0  SEM_SINALIZACAO_PELO_MODELO    424255                96.75
1            1      ENCAMINHAR_PARA_ANALISE     14255                 3.25


Exemplo dos resultados:


        ID  ESCORE_CLASSE_1  SINALIZACAO                         ACAO
0  5008804         0.066831            0  SEM_SINALIZACAO_PELO_MODELO
1  5008805         0.066831            0  SEM_SINALIZACAO_PELO_MODELO
2  5008806         0.029156            0  SEM_SINALIZACAO_PELO_MODELO
3  5008808         0.012367            0  SEM_SINALIZACAO_PELO_MODELO
4  5008809         0.012367            0  SEM_SINALIZACAO_PELO_MODELO
5  5008810         0.012367            0  SEM_SINALIZACAO_PELO_MODELO
6  5008811         0.012367            0  SEM_SINALIZACAO_PELO_MODELO
7  5008812         0.028960            0  SEM_SINALIZACAO_PELO_MODELO
8  5008813         0.028960            0  SEM_SINALIZACAO_PELO_MODELO
9  5008814         0.028960            0  SEM_SINALIZACAO_PELO_MODELO


Clientes com maiores escores entre os sinalizados:


             ID  ESCORE_CLASSE_1  SINALIZACAO                     ACAO
434432  5021303         0.987308            1  ENCAMINHAR_PARA_ANALISE
434733  5142972         0.987210            1  ENCAMINHAR_PARA_ANALISE
434484  5035868         0.985965            1  ENCAMINHAR_PARA_ANALISE
434649  5105739         0.985872            1  ENCAMINHAR_PARA_ANALISE
434552  5058350         0.984782            1  ENCAMINHAR_PARA_ANALISE
434739  5143578         0.984734            1  ENCAMINHAR_PARA_ANALISE
48639   5149189         0.984370            1  ENCAMINHAR_PARA_ANALISE
434758  5149190         0.984370            1  ENCAMINHAR_PARA_ANALISE
48638   5149188         0.984370            1  ENCAMINHAR_PARA_ANALISE
48640   5149191         0.984370            1  ENCAMINHAR_PARA_ANALISE
48641   5149192         0.984370            1  ENCAMINHAR_PARA_ANALISE
434517  5048379         0.983537            1  ENCAMINHAR_PARA_ANALISE
434629  5100111         0.982948            1  ENCAMINHAR_PARA_ANALISE
434689

## 7. Implicações de negócio

- **Uso recomendado: triagem.** O modelo ordena os clientes por risco e indica quem encaminhar para análise adicional. Não deve aprovar ou negar crédito sozinho.
- **Ganho real, mas moderado.** Average Precision de 0,12 contra 0,017 de um palpite aleatório (cerca de 7 vezes) e ROC-AUC de 0,62. Com o limiar 0,50, cerca de 1 em cada 9 sinalizações é um atraso grave e cerca de 1 em cada 5 atrasos graves é identificado (72 de 370 nos testes externos).
- **O que mais pesa:** idade, renda e tempo de emprego, seguidos de ocupação, estado civil e tipo de renda. Variáveis de contato (e-mail, telefone) e número de filhos pouco ajudam.
- **O limiar é uma escolha de custo.** Subir o recall para ~54% exige analisar ~40% da carteira; o limiar 0,50 analisa ~2,8%. A instituição deve definir o custo de uma análise e o de um atraso antes de fixar o limiar.
- **Limites.** Só dados cadastrais; poucos eventos (370 em 21.731); perfis iguais com desfechos opostos; resultados exploratórios, sem escores calibrados e sem confirmação de que os dados cadastrais estariam disponíveis antes da janela do alvo.
- **Próximos passos.** Incluir variáveis comportamentais, calibrar o limiar por custo, explorar mais modelos e monitorar desempenho e deriva em produção.